# Readme E-Manuscripta

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Manuscripta-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Mit Stand vom Juli 2023 gibt es 22 Objekte in E-Manuscripta der ZHB. Aufgrund der geringen Menge und der diversen Metadatenquellen, wird eine Excel-Datei händisch ergänzt mit den in Alma fehlenden Daten (bspw. DOI). 


## Basiskonfiguration

Die E-Manuscripta-Signaturen setzen sich aus dem E-Manuscripta DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI. 


### Config.py

Beispieldaten für ZHB E-Manuscripta. Anpassungen können in der config.py vorgenommen werden. 

    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Manuscripta'
    collection_id = 'zhb_emanuscripta'
    last_changed = 'yyyy-mm-dd'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    

### Import file

Der Import basiert auf einer Excel-Datei, in welcher die HAN-Nummern abgelegt sind, die in vielen Fällen für die Dateibezeichnungen verwendet werden. Diese Datei kann mit wenig manuellem Aufwand aus Alma extrahiert werden. Die E-Manuscripta befinden sich im Alma-Set e-manuscripta_dlza_heka. Sie wurde durch die DOI und Dateipfade ergänzt. Da die Sammlung überschaubar ist, hält sich der zeitliche Aufwand dafür in Grenzen.

Alternativ könnten die E-Manuscripta auch via OAI-Abfrage aus der Zenodo-OAI-Schnittstelle abgeholt werden. Sie befinden sich in folgender community:
https://zenodo.org/communities/lara_e-manuscripta

Weitere Infos: https://developers.zenodo.org/#oai-pmh


### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert (directory 'fulldump').

### Metadaten aus Alma (SRU, marcxml)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter signature.xml


### Datenobjekte 
Die E-Manuscripta-Zipkapseln sind alle nach folgender Struktur benannt:

    HAN-Nummer _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiel:

    000218785_20150312T000312_master_ver1.zip

Ausnahme: 

"Der ächte Eidsgenoss, eine wöchtentliche Sittenschrift. Zweyter Jahrgang / hrsg. von Johann Jakob Spreng", https://dx.doi.org/10.7891/e-manuscripta-108732 (keine HAN-Nummer vorhanden => DOI).

Die E-Codices-Objekte liegen auf G:\ZHB-Sosa_Digital\digital unter folgenden Pfaden:

    G:\ZHB-Sosa_Digital\digital\Pp
    G:\ZHB-Sosa_Digital\digital\Ms
    

Der aktuelle Dateipfad ist in der Excel-Datei in Spalte "filepath" abgelegt und wird auch in die Infojson unter 'additional' abgelegt. 


### TODO create Befehle erstellen für gocfl 

Die gocfl create Befehle für die E-Manuscripta-Sammlung (alle Objekte) werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt mit der Signature. 

Muster:

    gocfl create P:/temp/archiv P:/temp/testdata/zhb_erara_86774/object metadata:p:/temp/testdata/zhb_erara_86774/metadata --config p:/temp/config/gocfl.toml -i "zhb_erara_86774" --ext-NNNN-metafile-source p:/temp/testdata/zhb_erara_86774.json 

In [1]:
import json
import config
import pandas as pd
import requests

# Read the Excel file into a pandas DataFrame
input_file = "e-manuscripta.xlsx"
df = pd.read_excel(input_file)

# Convert the DataFrame to a list of dictionaries

completeSet = []

for _, row in df.iterrows():
    
    infoSet = {
        'additional': row['Path'].replace('\\','/'),
        'address': config.address, 
        'collection': config.collection,
        'collection_id': config.collection_id ,
        'created': str(row['Date']), 
        'identifiers': [str(row['MMS ID']), row['DOI'], row['HAN number'], row['Call number']],
        'ingest_workflow': config.ingest_workflow, 
        'keywords': config.keywords, 
        'last_changed': config.last_changed,
        'organisation' : config.organisation,
        'organisation_id' : config.organisation_id,
        'references' : ['https://doi.org/'+row['DOI'], 'https://rzs.swisscovery.slsp.ch/permalink/41SLSP_RZS/ldslj8/alma'+str(row['MMS ID'])],
        'signature': config.signature+row['DOI'].replace('.','_').replace('/','_'),
        'sets' : config.sets,
        'title' : row['Title'],
        'user' : row['Creator']      
    }
    
    signature = infoSet['signature']
    print("Signature:",signature)
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    
    infofile = f"info/{signature}.json"
    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
    
    # get metadata from Alma OAI as MARCXML
    alma_id = str(row['MMS ID'])   
    sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
    query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={alma_id}"
    response = requests.get(query)
    if response.status_code != 200:
        raise Exception(f"SRU request failed with status code {response.status_code}")

    # Save the response content (MARCXML) to a file
    metafile = f"metadata/{signature}.xml"
    with open(metafile, 'wb') as file:
        file.write(response.content)
        print(f"Record with ID {alma_id} saved as {metafile}")    


# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "fulldump/emanuscripta_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nfulldump written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "fulldump/emanuscripta_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved fulldump in excel file as {fullexcelfile}")


Signature: zhb_10_7891_e-manuscripta-24104
info.json saved as info/zhb_10_7891_e-manuscripta-24104.json
Record with ID 9914249335105505 saved as metadata/zhb_10_7891_e-manuscripta-24104.xml
Signature: zhb_10_7891_e-manuscripta-24479
info.json saved as info/zhb_10_7891_e-manuscripta-24479.json
Record with ID 9914249334005505 saved as metadata/zhb_10_7891_e-manuscripta-24479.xml
Signature: zhb_10_7891_e-manuscripta-24545
info.json saved as info/zhb_10_7891_e-manuscripta-24545.json
Record with ID 9914249332505505 saved as metadata/zhb_10_7891_e-manuscripta-24545.xml
Signature: zhb_10_7891_e-manuscripta-25099
info.json saved as info/zhb_10_7891_e-manuscripta-25099.json
Record with ID 9914249332405505 saved as metadata/zhb_10_7891_e-manuscripta-25099.xml
Signature: zhb_10_7891_e-manuscripta-23616
info.json saved as info/zhb_10_7891_e-manuscripta-23616.json
Record with ID 9914249331605505 saved as metadata/zhb_10_7891_e-manuscripta-23616.xml
Signature: zhb_10_7891_e-manuscripta-24491
info.js